# 02 — Querying and traversing the graph

Entries refer to each other by id, not by object reference. You traverse the
graph through collections (`kl.pathways`, `kl.behaviors`, `kl.impacts`, …) and
pathway views (`kl.pathway(...)`).

In [ ]:
from pathlib import Path


def repo_root() -> Path:
    """The checkout holding ``entries/``, whether Jupyter started in ``notebooks/`` or the root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "entries").is_dir() and (candidate / "src" / "coralmap").is_dir():
            return candidate
    raise FileNotFoundError("could not find the knowledge-library checkout")


REPO = repo_root()
REPO.name

In [ ]:
from coralmap import load

kl = load(REPO)
kl

## Lookup by either identifier

Accessors take the machine id (`kl-000024`) or the `display_id`
(`Impact-12`). The typed accessors (`kl.impact`, `kl.behavior`, `kl.pathway`)
raise `KeyError` when the key names an entry of another type; `kl.entry`
accepts any type.

In [ ]:
impact = kl.impact("Impact-12")
print(impact.id, impact.display_id, impact.label)
print(impact.definition[:240], "…")

same = kl.entry("kl-000024")
assert same.id == impact.id

In [ ]:
try:
    kl.impact("Pathway-5")
except KeyError as error:
    print(error)

## Collections and `filter`

Each keyword names a field on the entry or its `body`, and criteria are ANDed.
Enum fields match their string value, so `composition_type="omission"` works
without importing `CompositionType`. A keyword that names no field raises
`KeyError` instead of returning an empty result.

In [ ]:
print(kl.pathways)
print("conjunctive:", len(kl.pathways.filter(composition_type="conjunctive")))
print("omission:   ", len(kl.pathways.filter(composition_type="omission")))
print("direct:     ", len(kl.pathways.filter(composition_type="direct")))

In [ ]:
try:
    kl.pathways.filter(shap="omission")
except KeyError as error:
    print(error)

## Pathways leading to an impact

`kl.pathways.leading_to(impact)` returns the pathways with a direct edge to
that impact. Pathway-to-pathway composition is not followed.

In [ ]:
to_displaced_autonomy = kl.pathways.leading_to("Impact-12")
print(to_displaced_autonomy)
for pathway in to_displaced_autonomy:
    print(f"  {pathway.display_id:12}  {pathway.label}")

## Pathways involving a behavior

`involving()` goes the other way: the pathways in which a behavior is an
observation.

In [ ]:
behavior = kl.behavior("AiBehavior-1")
print(behavior.display_id, behavior.label)

implicated = kl.pathways.involving(behavior)
print(implicated)
for pathway in implicated:
    print(f"  {pathway.display_id:12}  {pathway.label}")

## Pathway views resolve ids to entries

`kl.pathway("Pathway-5")` returns a `PathwayView`. Its `.observations`,
`.impacts`, `.preconditions` and `.mechanisms` yield entries, not id strings.
`.observations` are the parts that make up the pattern in the conversation;
`.preconditions` are the parts that gate whether the pathway applies.
`.counterpart_owed` is authored prose describing what a well-handled version
would look like, or `None` where a contrast entry states it instead.

In [ ]:
view = kl.pathway("Pathway-5")
print(view)
print(view.entry.label)
print("composition_type ", view.entry.body.composition_type)
print("counterpart_owed ", view.counterpart_owed)
print()
print("observations")
for entry in view.observations:
    print(f"  {entry.display_id:16}  {entry.label}")
print("impacts")
for entry in view.impacts:
    print(f"  {entry.display_id:16}  {entry.label}")
print("preconditions")
for entry in view.preconditions:
    print(f"  {entry.display_id:16}  {getattr(entry, 'label', entry.id)}")
print("mechanisms")
for entry in view.mechanisms:
    print(f"  {entry.display_id:16}  {entry.label}")

## Entries are frozen

Assigning to a field raises `pydantic.ValidationError`. `public_fields()`
returns the entry as a dict without its `_`-prefixed internal keys.

In [ ]:
from pydantic import ValidationError

print(type(impact).__name__, "frozen:", impact.model_config.get("frozen"))
print(list(impact.public_fields())[:8])
try:
    impact.label = "rewritten"
except ValidationError as error:
    print(type(error).__name__, error.errors()[0]["type"])

Next: [03_models_contract_validation.ipynb](03_models_contract_validation.ipynb) —
the schema, the consumer contract, graph checks, and sources.
